In [ ]:
import re
import requests
import pandas as pd

archive_url = "https://openaq-data-archive.s3.amazonaws.com"
api_path = "Tables/dbo/openaq"
archive_path = "Tables/dbo/openaq_archive"
year = 2023
months = ["01", "02", "03"]

In [ ]:
rows = spark.read.format("delta").load(api_path).select("location_id").distinct().collect()
locations = [r["location_id"] for r in rows]

print(len(locations))

In [ ]:
keys = []
for loc in locations:
    for month in months:
        prefix = f"records/csv.gz/locationid={loc}/year={year}/month={month}/"
        xml = requests.get(archive_url, params={"list-type": 2, "prefix": prefix}).text
        keys = keys + re.findall("<Key>(.*?)</Key>", xml)

print(len(keys))

In [ ]:
df = pd.concat([pd.read_csv(archive_url + "/" + key) for key in keys])

print(len(df))

In [ ]:
spark.createDataFrame(df).write.format("delta").mode("overwrite").save(archive_path)